In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from scipy import stats
import numpy as np
import numpy.linalg as la
import math
from scipy.stats import norm


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/HW5_data.csv')
display(df.head())

,y,x1,x2,x3,x4,x5
0,-2.825707,-1.953838,-2.353305,1.858994,0.031193,0.031284
1,-2.347603,-0.615022,-0.074221,0.768311,-0.233615,-0.373178
2,0.546866,0.487606,1.285303,1.162639,-0.142936,-0.884465
3,-0.265539,-2.334235,-2.853869,-0.976867,-0.487647,-0.025365
4,-0.907260,0.361595,-0.320416,0.692993,0.482218,2.401161


## Problem 1

### 1.
Let $\varepsilon\sim N(0,I_n)$ (independent standard normals), and let $P\in\mathbb{R}^{n\times n}$ be orthogonal, i.e. $P'P=PP'=I_n$. Define
$$\nu = P\varepsilon .$$

Then
$$\mathbb{E}[\nu]=\mathbb{E}[P\varepsilon]=P\,\mathbb{E}[\varepsilon]=0,$$
and
$$\mathrm{Cov}(\nu)=\mathrm{Cov}(P\varepsilon)=P\,\mathrm{Cov}(\varepsilon)\,P'$$
$$= P I_n P' = PP' = I_n.$$
So for each component $i=1,\dots,n$,
$$\mathbb{E}(\nu_i)=0,\qquad \mathrm{Var}(\nu_i)=1.$$

In [4]:
X = df[[f"x{i}" for i in range(1, 6)]].to_numpy()
n, p = X.shape

# SVD: X = U Σ V^T
U, s, VT = np.linalg.svd(X, full_matrices=True)

# Match the notation in the prompt:
P = U.T          # n×n orthogonal
Q = VT           # p×p orthogonal (so Q' = V = VT.T)
Lambda2 = np.diag(s**2)

# Verify Q X'X Q' = Λ^2
print("||Q X'X Q' - Λ^2|| =", np.linalg.norm(Q @ (X.T @ X) @ Q.T - Lambda2))

# Monte Carlo: v = P ε should have mean 0 and variance 1 componentwise
rng = np.random.default_rng(0)
m = 20000
eps = rng.standard_normal((m, n))
v = eps @ P.T  # row-wise version of v = P ε

print("max |mean(v_i)| =", np.max(np.abs(v.mean(axis=0))))
print("min var(v_i)   =", np.min(v.var(axis=0)))
print("max var(v_i)   =", np.max(v.var(axis=0)))

||Q X'X Q' - Λ^2|| = 5.574490399420855e-13
max |mean(v_i)| = 0.023605015041270185
min var(v_i)   = 0.9702628227517754
max var(v_i)   = 1.0270576134792788


### 2.
Assume $\gamma = Q\beta$. Since $Q$ is orthogonal, $Q'Q=I_p$, so $\beta = Q'\gamma$. Then
$$PX\beta = PX(Q'\gamma) = (PXQ')\gamma = D\gamma,$$
because you are given $PXQ' = D$.

Now start from the original model
$$y = X\beta + \sigma \varepsilon,$$
left-multiply by $P$ and define $z = Py$, $\nu = P\varepsilon$:
$$z = Py = PX\beta + \sigma P\varepsilon = D\gamma + \sigma \nu.$$
With
$$D=\begin{pmatrix}\Lambda\\0\end{pmatrix},\qquad \Lambda=\mathrm{diag}(\lambda_1,\dots,\lambda_p),$$
this becomes componentwise:
$$z_i=
\begin{cases}
\lambda_i \gamma_i + \sigma \nu_i, & i=1,\dots,p,\ [4pt]
\sigma \nu_i, & i=p+1,\dots,n.
\end{cases}$$

In [5]:
# Construct D = (Λ; 0)
D = np.vstack([np.diag(s), np.zeros((n - p, p))])

# Verify PXQ' = D
print("||P X Q' - D|| =", np.linalg.norm(P @ X @ Q.T - D))

# Pick an arbitrary beta and verify PX beta = D gamma
beta = rng.normal(size=(p, 1))
gamma = Q @ beta
lhs = P @ X @ beta
rhs = D @ gamma
print("||PXβ - Dγ|| =", np.linalg.norm(lhs - rhs))

||P X Q' - D|| = 2.933377278743338e-14
||PXβ - Dγ|| = 3.572289414099341e-14


### 3.
In the linear model $y=X\beta+\sigma\varepsilon$ with $\varepsilon\sim N(0,I_n)$, the OLS estimator is
$$\hat\beta=(X'X)^{-1}X'y,$$
and its covariance matrix is
$$\mathrm{Cov}(\hat\beta)=\sigma^2 (X'X)^{-1}.$$
Therefore the sum of the component variances is the trace:
$$\sum_{j=1}^p \mathrm{Var}(\hat\beta_j)
=\mathrm{tr}(\mathrm{Cov}(\hat\beta))
=\sigma^2\,\mathrm{tr}\big((X'X)^{-1}\big).$$

From the given decomposition $QX'XQ'=\Lambda^2$ with $\Lambda=\mathrm{diag}(\lambda_1,\dots,\lambda_p)$, we have
$$X'X=Q'\Lambda^2 Q
\quad\Longrightarrow\quad
(X'X)^{-1}=Q'\Lambda^{-2}Q.$$
Using invariance of trace under cyclic permutation:
$$\mathrm{tr}\big((X'X)^{-1}\big)
=\mathrm{tr}(Q'\Lambda^{-2}Q)
=\mathrm{tr}(\Lambda^{-2}QQ')
=\mathrm{tr}(\Lambda^{-2})
=\sum_{j=1}^p \frac{1}{\lambda_j^2}.$$
So
$$\boxed{\;\sum_{j=1}^p \mathrm{Var}(\hat\beta_j)=\sigma^2\sum_{j=1}^p \frac{1}{\lambda_j^2}\;}$$

### 4.
find $\hat\gamma$

In [6]:
y = df["y"].to_numpy().reshape(-1, 1)
X = df[[f"x{i}" for i in range(1, 6)]].to_numpy()
n, p = X.shape

# OLS beta-hat
beta_hat = np.linalg.inv(X.T @ X) @ (X.T @ y)

# Build Q from SVD: X = U Σ V^T, then X'X = V Σ^2 V^T, so choose Q = V^T
U, s, VT = np.linalg.svd(X, full_matrices=False)
Q = VT  # Q X'X Q' = diag(s^2)

gamma_hat = Q @ beta_hat
gamma_hat

array([[ 0.88428498],
       [ 0.06425593],
       [-1.66167113],
       [ 1.41376844],
       [ 0.30486612]])

### 5.
Find $m$ and the corresponding $\hat\beta_{\text{pca}}$

In [9]:
lam = s  # these are the λ_j
ratio = np.cumsum(lam**2) / np.sum(lam**2)
ratio
print("So the smallest m with ratio >0.9 is: ", 4, " and m = ", 4)

So the smallest m with ratio >0.9 is:  4  and m =  4


In [10]:
m = 4
V = VT.T

# z = U' y (thin U is enough for first p components)
z = U.T @ y  # p×1

gamma_hat_trunc = np.zeros((p, 1))
gamma_hat_trunc[:m] = z[:m] / lam[:m, None]   # keep first m, zero out rest

beta_pca = V @ gamma_hat_trunc
beta_pca

array([[ 1.57021141],
       [-0.96725821],
       [ 0.11719105],
       [ 0.05748972],
       [-1.45869454]])

## Problem 2

### 1.
Write down the posterior density for β and note that this is always of the form prior times likelihood.

Model:
$y_i\mid \beta \sim N(x_i\beta,\sigma^2),\quad i=1,\dots,n,$
with summaries $\sum_{i=1}^n x_i=0,\ \sum_{i=1}^n x_i^2=n,\ \sum_{i=1}^n x_i y_i=\gamma.$
Prior (spike-and-slab):
$$\pi(\beta)=0.5\,\delta_0(\beta)+0.5\,N(\beta\mid 0,\tau^2),$$
where $\delta_0$ is a point mass at 0.

Likelihood as a function of $\beta$

Let $S_{yy}=\sum_{i=1}^n y_i^2$. Then
$$\sum_{i=1}^n (y_i-x_i\beta)^2
= \sum y_i^2 -2\beta\sum x_i y_i + \beta^2 \sum x_i^2
= S_{yy}-2\beta\gamma+n\beta^2.$$
So
$$L(\beta)\equiv p(y\mid \beta)
=(2\pi\sigma^2)^{-n/2}\exp\!\left[-\frac{1}{2\sigma^2}\left(S_{yy}-2\beta\gamma+n\beta^2\right)\right].$$

Posterior (always “prior $\times$ likelihood”)

$$\pi(\beta\mid y)\ \propto\ \pi(\beta)\,L(\beta).$$
Because the prior is a mixture, the posterior is also a mixture:
$$\boxed{\ \pi(\beta\mid y)=w_0\,\delta_0(\beta)+(1-w_0)\,N(\beta\mid \mu_n, v_n)\ }$$
where the slab posterior parameters come from Normal–Normal conjugacy:
$$v_n=\left(\frac{n}{\sigma^2}+\frac{1}{\tau^2}\right)^{-1},\qquad
\mu_n=v_n\left(\frac{\gamma}{\sigma^2}\right).$$

The posterior spike weight is (equal prior weights 0.5 cancel)
$$w_0=\frac{L(0)}{L(0)+m(y)},$$
where $m(y)$ is the slab marginal likelihood from part (2) below.

### 2.
Marginal likelihood

You are asked to compute
$$m(y)=\int \left[\prod_{i=1}^n N(y_i\mid x_i\beta,\sigma^2)\right] N(\beta\mid 0,\tau^2)\,d\beta.$$

Using the likelihood expression above,
$$m(y)=(2\pi\sigma^2)^{-n/2}\exp\!\left(-\frac{S_{yy}}{2\sigma^2}\right)
\int \frac{1}{\sqrt{2\pi\tau^2}}
\exp\!\left\{-\frac{1}{2}\Big[\Big(\frac{n}{\sigma^2}+\frac{1}{\tau^2}\Big)\beta^2-2\Big(\frac{\gamma}{\sigma^2}\Big)\beta\Big]\right\}\,d\beta.$$

Let
$$A=\frac{n}{\sigma^2}+\frac{1}{\tau^2},\qquad B=\frac{\gamma}{\sigma^2}.$$
Complete the square:
$$A\beta^2-2B\beta=A(\beta-B/A)^2-\frac{B^2}{A}.$$
Then the integral is a Gaussian integral:
$$\int \frac{1}{\sqrt{2\pi\tau^2}} \exp\!\left[-\frac{A}{2}(\beta-B/A)^2\right]d\beta
=\frac{1}{\sqrt{\tau^2 A}}.$$
and we keep the leftover $\exp(B^2/(2A))$. Therefore:
$$\boxed{
m(y)= (2\pi\sigma^2)^{-n/2}\exp\!\left(-\frac{S_{yy}}{2\sigma^2}\right)\cdot
\frac{1}{\sqrt{1+n\tau^2/\sigma^2}}\cdot
\exp\!\left(\frac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}\right)
}$$
(using $\tau^2A = 1+n\tau^2/\sigma^2$ and $\frac{B^2}{2A}=\frac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}$).

Useful simplification for the spike weight $w_0$

Since
$$L(0)=(2\pi\sigma^2)^{-n/2}\exp\!\left(-\frac{S_{yy}}{2\sigma^2}\right),$$
we have
$$\frac{m(y)}{L(0)}=
\frac{1}{\sqrt{1+n\tau^2/\sigma^2}}\exp\!\left(\frac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}\right),$$
so
$$\boxed{
w_0=\frac{1}{1+\dfrac{1}{\sqrt{1+n\tau^2/\sigma^2}}
\exp\!\left(\dfrac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}\right)}
}.$$


### 3.
Show the posterior mixture form and find $\nu,\psi$

We already have the spike-and-slab prior
$\pi(\beta)=\tfrac12\,\mathbf 1(\beta=0)+\tfrac12\,N(\beta\mid 0,\tau^2),$
and likelihood
$$L(\beta)=\prod_{i=1}^n N(y_i\mid x_i\beta,\sigma^2).$$

From earlier algebra,
$$\sum_{i=1}^n (y_i-x_i\beta)^2
= S_{yy}-2\gamma\beta+n\beta^2,
\quad
S_{yy}=\sum y_i^2,\ \gamma=\sum x_i y_i,\ \sum x_i^2=n.$$
So (up to constants in $\beta$)
$$L(\beta)\propto \exp\!\left[-\frac{1}{2\sigma^2}(n\beta^2-2\gamma\beta)\right].$$

Spike part

At $\beta=0$,
$$L(0)=\prod_{i=1}^n N(y_i\mid 0,\sigma^2).$$
Thus the spike contribution to the unnormalized posterior is
$$\Big\{\prod_{i=1}^n N(y_i\mid 0,\sigma^2)\Big\}\,\mathbf 1(\beta=0).$$

Slab part (complete the square)

The slab contribution is
$$L(\beta)\,N(\beta\mid 0,\tau^2)
\propto
\exp\!\left\{
-\frac{1}{2}\Big[
\Big(\frac{n}{\sigma^2}+\frac{1}{\tau^2}\Big)\beta^2
-2\Big(\frac{\gamma}{\sigma^2}\Big)\beta
\Big]\right\}.$$
This is proportional to a Normal density $N(\beta\mid \nu,\psi^2)$, where
$$\boxed{\ \psi^2=\left(\frac{n}{\sigma^2}+\frac{1}{\tau^2}\right)^{-1}=\frac{\sigma^2\tau^2}{n\tau^2+\sigma^2}\ },
\qquad
\boxed{\ \nu=\psi^2\left(\frac{\gamma}{\sigma^2}\right)=\frac{\tau^2\,\gamma}{n\tau^2+\sigma^2}\ }.$$

The missing multiplicative constant (in $\beta$) from this completion-of-square is exactly the marginal likelihood
$m(y)=\int L(\beta)\,N(\beta\mid 0,\tau^2)\,d\beta,$
which you computed in part (2). Therefore the posterior can be written (up to a global normalizing constant) as:
$$\boxed{
\pi(\beta\mid y_1,\dots,y_n)\ \propto\
\Big\{\prod_{i=1}^n N(y_i\mid 0,\sigma^2)\Big\}\mathbf 1(\beta=0)
\;+\;
m(y_1,\dots,y_n)\,N(\beta\mid \nu,\psi^2)
}.$$

###4.
Write down $\pi(\beta=0\mid y_1,\dots,y_n)$

Let
$$L_0 \equiv \prod_{i=1}^n N(y_i\mid 0,\sigma^2).$$
Then the posterior mass at 0 is the mixture weight:
$$\boxed{
\pi(\beta=0\mid y)=\frac{L_0}{L_0+m(y)}
}.$$
(Here the prior weights are both 0.5, so they cancel in the ratio.)

Equivalently,
$$\pi(\beta=0\mid y)=\frac{1}{1+\dfrac{m(y)}{L_0}}.$$

###5.
Significance of $\dfrac{m(y)}{\prod_{i=1}^n N(y_i\mid 0,\sigma^2)}$ and show it $\to\infty$ as $\gamma^2$ grows

Significance

$$\frac{m(y)}{\prod_{i=1}^n N(y_i\mid 0,\sigma^2)}
=
\frac{p(y\mid \text{slab model: }\beta\sim N(0,\tau^2))}
{p(y\mid \text{spike model: }\beta=0)}.$$
So it is the Bayes factor comparing:
* $H_0:\beta=0$ (spike / null model), versus
* $H_1:\beta\sim N(0,\tau^2)$ (slab / alternative with $\beta$ integrated out).
It controls the posterior spike probability:
$$\pi(\beta=0\mid y)=\frac{1}{1+\text{BF}_{10}},
\quad \text{where } \text{BF}_{10}=\frac{m(y)}{L_0}.$$

Show it diverges as $\gamma^2\to\infty$

From part (2), we had
$$m(y)=L_0\cdot \frac{1}{\sqrt{1+n\tau^2/\sigma^2}}\,\
\exp\!\left(\frac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}\right).$$
Therefore
$$\boxed{
\frac{m(y)}{L_0}
=
\frac{1}{\sqrt{1+n\tau^2/\sigma^2}}\,\
\exp\!\left(\frac{\tau^2\gamma^2}{2\sigma^2(n\tau^2+\sigma^2)}\right)
}.$$
The prefactor $\frac{1}{\sqrt{1+n\tau^2/\sigma^2}}$ is a positive constant (given $n,\sigma,\tau$). The exponential term is
$$\exp(c\,\gamma^2)
\quad \text{with}\quad
c=\frac{\tau^2}{2\sigma^2(n\tau^2+\sigma^2)} > 0.$$
Hence as $\gamma^2\to\infty$,
$$\frac{m(y)}{L_0}\to\infty.$$
So the data increasingly favor the slab model over $\beta=0$, and consequently
$$\pi(\beta=0\mid y)=\frac{1}{1+m(y)/L_0}\to 0.$$